In [2]:
!apt-get update -y && apt-get install -y iverilog
!pip -q install requests

import os, re, json, textwrap, subprocess
import requests

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,916 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,115 kB]
Get:14 h

In [3]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   9891      0 --:--:-- --:--:-- --:--:--  9924


In [4]:
SPEC = """
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Data (3 bits)
    - Outputs:
        - Sequence found

While enabled, it should detect the following sequence of binary input values:
    - 0b001
    - 0b101
    - 0b110
    - 0b000
    - 0b110
    - 0b110
    - 0b011
    - 0b101

How would I write a design that meets these specifications?
"""

PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module — no testbench.


Spec:
{SPEC}
""".strip()

In [5]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model
model = genai.GenerativeModel('gemini-2.5-flash')
completion = model.generate_content(PROMPT)
print(completion.text)

```verilog
module sequence_detector (
    input wire clock,
    input wire n_reset, // Active-low reset
    input wire [2:0] data,
    output reg sequence_found
);

// State parameter definitions (using 4 bits for future expandability, 3 bits would suffice for 8 states)
localparam S_IDLE      = 4'b0000;
localparam S_001_1     = 4'b0001; // First element: 0b001
localparam S_101_1     = 4'b0010; // Second element: 0b101
localparam S_110_1     = 4'b0011; // Third element: 0b110
localparam S_000_1     = 4'b0100; // Fourth element: 0b000
localparam S_110_2     = 4'b0101; // Fifth element: 0b110
localparam S_110_3     = 4'b0110; // Sixth element: 0b110
localparam S_011_1     = 4'b0111; // Seventh element: 0b011

reg [3:0] current_state;
reg [3:0] next_state;

// Synchronous state update logic
always_ff @(posedge clock or negedge n_reset) begin
    if (!n_reset) begin
        current_state <= S_IDLE;
    end else begin
        current_state <= next_state;
    end
end

// Combinational next st

In [6]:
def extract_sv_code(text: str) -> str:
    """Extract verilog code from a fenced ```verilog ... ``` block."""
    import re
    pattern = r"```verilog\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: try generic code block
    pattern = r"```\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Last resort: return raw text
    return text.strip()


In [7]:
sv_code = extract_sv_code(completion.text)
print("EXTRACTED SYSTEMVERILOG (preview)")
print(sv_code)

EXTRACTED SYSTEMVERILOG (preview)
module sequence_detector (
    input wire clock,
    input wire n_reset, // Active-low reset
    input wire [2:0] data,
    output reg sequence_found
);

// State parameter definitions (using 4 bits for future expandability, 3 bits would suffice for 8 states)
localparam S_IDLE      = 4'b0000;
localparam S_001_1     = 4'b0001; // First element: 0b001
localparam S_101_1     = 4'b0010; // Second element: 0b101
localparam S_110_1     = 4'b0011; // Third element: 0b110
localparam S_000_1     = 4'b0100; // Fourth element: 0b000
localparam S_110_2     = 4'b0101; // Fifth element: 0b110
localparam S_110_3     = 4'b0110; // Sixth element: 0b110
localparam S_011_1     = 4'b0111; // Seventh element: 0b011

reg [3:0] current_state;
reg [3:0] next_state;

// Synchronous state update logic
always_ff @(posedge clock or negedge n_reset) begin
    if (!n_reset) begin
        current_state <= S_IDLE;
    end else begin
        current_state <= next_state;
    end
end

/

In [8]:
sv_code = r"""
module sequence_detector (
    input wire clk,
    input wire rst_n, // Active-low reset
    input wire [2:0] data,
    output reg sequence_found
);

    // State encoding (9 states, need 4 bits)
    localparam [3:0]
        IDLE        = 4'd0,
        S1_001      = 4'd1,
        S2_101      = 4'd2,
        S3_110_1    = 4'd3,
        S4_000      = 4'd4,
        S5_110_2    = 4'd5,
        S6_110_3    = 4'd6,
        S7_011      = 4'd7,
        S8_FINAL    = 4'd8; // State when the full sequence is detected

    // Current state and next state registers
    reg [3:0] current_state_reg;
    reg [3:0] next_state_logic;

    // State flip-flop
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            current_state_reg <= IDLE;
        end else begin
            current_state_reg <= next_state_logic;
        end
    end

    // Next state and output combinational logic
    always @(*) begin
        next_state_logic = current_state_reg; // Default: stay in current state
        sequence_found = 1'b0;                // Default: output not asserted

        case (current_state_reg)
            IDLE: begin
                if (data == 3'b001) begin
                    next_state_logic = S1_001;
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S1_001: begin
                if (data == 3'b101) begin
                    next_state_logic = S2_101;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1 if 001 seen
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S2_101: begin
                if (data == 3'b110) begin
                    next_state_logic = S3_110_1;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S3_110_1: begin
                if (data == 3'b000) begin
                    next_state_logic = S4_000;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S4_000: begin
                if (data == 3'b110) begin
                    next_state_logic = S5_110_2;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S5_110_2: begin
                if (data == 3'b110) begin
                    next_state_logic = S6_110_3;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S6_110_3: begin
                if (data == 3'b011) begin
                    next_state_logic = S7_011;
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S7_011: begin
                if (data == 3'b101) begin
                    next_state_logic = S8_FINAL;
                    sequence_found = 1'b1; // Assert output when final element matches
                end else if (data == 3'b001) begin
                    next_state_logic = S1_001; // Restart from S1
                end else begin
                    next_state_logic = IDLE;
                end
            end
            S8_FINAL: begin
                // Sequence just completed, reset to IDLE for next detection
                next_state_logic = IDLE;
            end
            default: begin
                next_state_logic = IDLE; // Fallback for undefined states
            end
        endcase
    end

endmodule
"""
filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)


In [9]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector_tb.v:10: error: port ``reset_n'' is not a port of dut.
1 error(s) during elaboration.


In [10]:
SPEC = """
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Data (3 bits)
    - Outputs:
        - Sequence found

While enabled, it should detect the following sequence of binary input values:
    - 0b001
    - 0b101
    - 0b110
    - 0b000
    - 0b110
    - 0b110
    - 0b011
    - 0b101

How would I write a design that meets these specifications?
"""
PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module — no testbench.
5) You MUST start your module with EXACTLY this header, copy it verbatim:

module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);

6) Do NOT rename any port. Use exactly: clk, reset_n, data, sequence_found
7) reset_n is active-low: reset when (!reset_n)
8) Use always @(posedge clk or negedge reset_n) for state flip-flop.

Spec:
{SPEC}
""".strip()

In [11]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash-lite')
completion = model.generate_content(PROMPT)


In [12]:
def extract_sv_code(text: str) -> str:
    import re
    pattern = r"```verilog\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    pattern = r"```\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

sv_code = extract_sv_code(completion.text)
print("EXTRACTED SYSTEMVERILOG (preview)")
print(sv_code)


filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)

!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

EXTRACTED SYSTEMVERILOG (preview)
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,
    input  wire [2:0]  data,
    output reg         sequence_found
);

    typedef enum logic [2:0] {
        IDLE,
        S001,
        S101_1,
        S110_1,
        S000,
        S110_2,
        S110_3,
        S011,
        S101_2,
        DONE
    } state_t;

    reg state_t current_state, next_state;

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= IDLE;
        end else begin
            current_state <= next_state;
        end
    end

    always @(*) begin
        next_state = current_state;
        sequence_found = 1'b0;

        case (current_state)
            IDLE: begin
                if (data == 3'b001) begin
                    next_state = S001;
                end
            end
            S001: begin
                if (data == 3'b101) begin
                    next_state = S101_1;
  

In [13]:
filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)

!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector.v:21: error: Enumeration name S101_2 has an inferred value that overflowed.
sequence_detector.v:21: error: Enumeration name S101_2 and IDLE have the same value: 3'd0
sequence_detector.v:21: error: Enumeration name DONE has an inferred value that overflowed.
sequence_detector.v:21: error: Enumeration name DONE and S001 have the same value: 3'd1
5 error(s) during elaboration.


In [14]:
PROMPT = f"""
You are an expert digital design engineer.

Task: Write synthesizable Verilog code that compiles with:
iverilog -g2012

CRITICAL OUTPUT RULES (you MUST follow these exactly):
1) Your ENTIRE response must be ONLY a single fenced code block: ```verilog ... ```
2) NO explanations, NO prose, NO comments outside the code block.
3) NO manual step-by-step solving. Output CODE only.
4) The code block must contain ONLY the module — no testbench.
5) You MUST start your module with EXACTLY these two lines, copy verbatim:

`timescale 1ns/1ps
module sequence_detector(

   Note: the backtick in `timescale is REQUIRED. Do not omit it.

CRITICAL IMPLEMENTATION RULES:
6) Use a single reg [3:0] state register. Do NOT define named parameters or localparams for states — use plain integers 0–7 directly in the case statement.
7) Use ONE sequential always @(posedge clk or negedge reset_n) block with case(state) for transitions. No next_state register.
8) Use ONE combinational always @(*) block ONLY for sequence_found output.
9) sequence_found must be asserted like this EXACTLY:
       if (state == 7 && data == 3'b101)
           sequence_found = 1;
       else
           sequence_found = 0;
10) The FSM only needs states 0–7. There is NO state 8. Detection happens combinationally in the output block.
11) On mismatch in any state, return to state 0.
12) reset_n is active-low: on (!reset_n), set state <= 0.

Spec:
{SPEC}
""".strip()

In [15]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')
completion = model.generate_content(PROMPT)


In [16]:
def extract_sv_code(text: str) -> str:
    import re
    pattern = r"```verilog\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    pattern = r"```\s*(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

sv_code = extract_sv_code(completion.text)
print("EXTRACTED SYSTEMVERILOG (preview)")
print(sv_code)


filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)

!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

EXTRACTED SYSTEMVERILOG (preview)
`timescale 1ns/1ps
module sequence_detector(
    input clk,
    input reset_n,
    input [2:0] data,
    output reg sequence_found
);

reg [3:0] state;

// State transition logic
always @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        state <= 0; // Reset to initial state
    end else begin
        case(state)
            0: begin // Waiting for 0b001
                if (data == 3'b001) state <= 1;
                else state <= 0; // Mismatch, stay in initial state
            end
            1: begin // Received 0b001, waiting for 0b101
                if (data == 3'b101) state <= 2;
                else state <= 0; // Mismatch, return to initial state
            end
            2: begin // Received 0b001, 0b101, waiting for 0b110
                if (data == 3'b110) state <= 3;
                else state <= 0; // Mismatch, return to initial state
            end
            3: begin // Received ..., 0b110, waiting for 0b000
   

In [19]:
sv_code = r"""
`timescale 1ns/1ps
module sequence_detector(
    input clk,
    input reset_n,
    input [2:0] data,
    output reg sequence_found
);

reg [3:0] state;

// State transition logic
always @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        state <= 0; // Reset to initial state
    end else begin
        case(state)
            0: begin // Waiting for 0b001
                if (data == 3'b001) state <= 1;
                else state <= 0; // Mismatch, stay in initial state
            end
            1: begin // Received 0b001, waiting for 0b101
                if (data == 3'b101) state <= 2;
                else state <= 0; // Mismatch, return to initial state
            end
            2: begin // Received 0b001, 0b101, waiting for 0b110
                if (data == 3'b110) state <= 3;
                else state <= 0; // Mismatch, return to initial state
            end
            3: begin // Received ..., 0b110, waiting for 0b000
                if (data == 3'b000) state <= 4;
                else state <= 0; // Mismatch, return to initial state
            end
            4: begin // Received ..., 0b000, waiting for 0b110
                if (data == 3'b110) state <= 5;
                else state <= 0; // Mismatch, return to initial state
            end
            5: begin // Received ..., 0b110, waiting for 0b110
                if (data == 3'b110) state <= 6;
                else state <= 0; // Mismatch, return to initial state
            end
            6: begin // Received ..., 0b110, waiting for 0b011
                if (data == 3'b011) state <= 7;
                else state <= 0; // Mismatch, return to initial state
            end
            7: begin // Received ..., 0b011, waiting for 0b101
                // If data matches 0b101, sequence_found will be asserted combinatorially.
                // In either case (match or mismatch), return to state 0 to look for next sequence.
                state <= 0;
            end
            default: begin // Should not happen with 3-bit state, but ensures full state space is covered
                state <= 0;
            end
        endcase
    end
end

// Output logic for sequence_found
always @(*) begin
    if (state == 7 && data == 3'b101)
        sequence_found = 1;
    else
        sequence_found = 0;
end

endmodule
"""
filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)


In [20]:
filename = "sequence_detector/sequence_detector.v"
with open(filename, "w") as f:
    f.write(sv_code)

!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

All test cases passed.
